In [ ]:
# Stop any existing session
try:
    spark.stop()
except:
    pass

In [ ]:
# Start a SparkSession
from pyspark.sql import Row, SparkSession

spark = (
    SparkSession.builder.appName("sample_chapter_idempotent").master("local[*]").getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# Connection properties
url = "jdbc:postgresql://postgres:5432/ecommerce"
properties = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}

## Data pipeline scripts should be re-runnable without creating duplicate or partial data (aka idempotent)

* **Idempotency**: If you run the code multiple times with the same input, the output should be the same. When storing the output in an external data store, it should not be duplicated or partial. Mathematically defined as `f(f(x)) = f(x)`
* To create an idempotent pipeline, there are a few criteria to satisfy.
  * **Atomicity**: A pipeline should only create one table.
  * **No side effects**: A pipeline should not affect any external data (variable or other) besides its output. (with the exception of logs)
* Within a pipeline, it is helpful to have individual functions corresponding to Extract, Transform, & Load, and they can be retried independently.
* Creating data transformations as functions makes the code:
  1. Easy to maintain and debug.
  2. Easy to test.
  3. Represent the reality of transforming data in a series of steps (represented as functions).
  4. Simpler to run in parallel (if your transformations are independent of historical data) with different inputs.

#### Example

* Let’s go over the `dim_customer_snapshot` pipeline. No matter how many times we run it with the same input (data source and time range), the output will be the same. We will not get duplicate rows or partial rows.

In [ ]:
from pyspark.sql import DataFrame, SparkSession

JDBC_URL = "postgre_connection_url"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}
TABLE_NAME = "local.warehouse.dim_customer_snapshot"


def extract(spark: SparkSession) -> dict[str, DataFrame]:
    customer_df = spark.read.jdbc(
        url=url, table="public.customer", properties=properties
    )
    customer_address_df = spark.read.jdbc(
        url=url, table="public.customer_address", properties=properties
    )
    return {"customer_df": customer_df, "customer_address_df": customer_address_df}


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    input_dfs["customer_df"].createOrReplaceTempView("customer")
    input_dfs["customer_address_df"].createOrReplaceTempView("customer_address")

    return spark.sql("""
        SELECT
            c.customer_id,
            c.email,
            c.full_name,
            c.phone,
            c.status,
            c.created_at,
            c.updated_at,
            COLLECT_LIST(
                STRUCT(
                    ca.is_default,
                    CONCAT(ca.line1, ', ', ca.city, ', ', ca.state, ', ', ca.country) AS address
                )
            ) AS addresses
        FROM customer c
        LEFT JOIN customer_address ca USING (customer_id)
        GROUP BY 1, 2, 3, 4, 5, 6, 7
    """)


def load(output_df: DataFrame) -> None:
    output_df.writeTo(TABLE_NAME).createOrReplace()


def run(spark: SparkSession) -> None:
    load(transform(extract(spark)))

In [ ]:
run(spark)

In [ ]:
spark.table("local.warehouse.dim_customer_snapshot").count()

In [ ]:
run(spark)

In [ ]:
spark.table("local.warehouse.dim_customer_snapshot").count()

#### Exercise [5 min]

Is the `fct_order_lines_incremental` pipeline idempotent? Why, why not?

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

TABLE_NAME = "local.warehouse.fct_order_lines_incremental"

JDBC_URL = "jdbc:postgresql://postgres:5432/ecommerce"
JDBC_PROPERTIES = {
    "user": "dataengineer",
    "password": "datapipeline",
    "driver": "org.postgresql.Driver",
}


def extract(
    spark: SparkSession,
    start_time: str,
    end_time: str,
) -> dict[str, DataFrame]:
    order_line_df = spark.read.jdbc(
        url=JDBC_URL,
        table="public.order_line",
        properties=JDBC_PROPERTIES,
    )
    return {
        "order_line": order_line_df.filter(
            (F.col("created_at") >= start_time) & (F.col("created_at") < end_time)
        )
    }


def transform(input_dfs: dict[str, DataFrame]) -> DataFrame:
    return input_dfs["order_line"].select(
        F.col("order_line_id"),
        F.col("order_id"),
        F.col("variant_id"),
        F.col("quantity"),
        F.col("unit_price"),
        F.col("discount_amt"),
        F.col("line_total"),
        F.col("created_at"),
        F.col("updated_at"),
        (F.col("discount_amt") > 0).alias("is_discounted"),
        (F.col("unit_price") - F.col("discount_amt")).alias("effective_unit_price"),
    )


def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        (
            output_df.writeTo(TABLE_NAME)
            .partitionedBy(F.partitioning.days("created_at"))
            .createOrReplace()
        )
    else:
        output_df.writeTo(TABLE_NAME).overwritePartitions()


def run(spark: SparkSession, start_time: str, end_time: str) -> None:
    load(transform(extract(spark, start_time, end_time)), spark)

* Is SCD2 with a `MERGE INTO` idempotent?

SCD2 code 

```python
def load(output_df: DataFrame, spark: SparkSession) -> None:
    if not spark.catalog.tableExists(TABLE_NAME):
        output_df.withColumn("valid_from", F.col("created_at")).withColumn(
            "valid_to", F.lit(None).cast("timestamp")
        ).withColumn("is_current", F.lit(True)).writeTo(TABLE_NAME).createOrReplace()
        return

    output_df.createOrReplaceTempView("staged")

    spark.sql(f"""
        WITH products_with_updates AS (
            SELECT b.*
            FROM staged b
            JOIN {TABLE_NAME} d
                ON b.product_id = d.product_id
            WHERE b.updated_at > d.updated_at
            AND d.is_current = true
        )
        MERGE INTO {TABLE_NAME} t
        USING (
            SELECT product_id AS join_key, * FROM staged
            UNION ALL
            SELECT NULL AS join_key, * FROM products_with_updates
        ) s
        ON t.product_id = s.join_key

        WHEN MATCHED
            AND t.is_current = true
            AND s.updated_at > t.updated_at
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to = s.updated_at

        WHEN NOT MATCHED
        THEN INSERT (
            product_id,
            product_name,
            brand,
            base_price,
            status,
            updated_at,
            created_at,
            valid_from,
            valid_to,
            is_current
        ) VALUES (
            s.product_id,
            s.product_name,
            s.brand,
            s.base_price,
            s.status,
            s.updated_at,
            s.created_at,
            s.updated_at,
            NULL,
            TRUE
        )

        WHEN NOT MATCHED BY SOURCE
            AND t.is_current = true
        THEN UPDATE SET
            t.is_current = false,
            t.valid_to = current_timestamp()
    """)
```